In [3]:
from importlib import reload

import numpy as np
import pyvista as pv
import xarray as xr

import skyvista as sv

In [55]:
ds = xr.open_dataset(
    "/home/cmdavis4/projects/carlee/skyvista/examples/MergedReflectivityQCComposite_00.50_20250906150000-20250906160000.nc"
).squeeze()
# Rename coordinates
ds = ds.drop_vars("time").rename_dims(
    {"valid_time": "time", "latitude": "y", "longitude": "x"}
)
# Pick one time and coarsen
ds = ds.isel({"time": 0, "x": slice(None, None, 10), "y": slice(None, None, 10)})
# Convert -999 to missing
ds["unknown"] = ds["unknown"].where(ds["unknown"] != -999.0, np.nan)
# ds['unknown'] = ds['unknown'].where(ds['unknown'] == -999., np.nan)
ds

<xarray.Dataset> Size: 988kB
Dimensions:         (y: 350, x: 700)
Coordinates:
  * latitude        (y) float64 3kB 54.99 54.9 54.8 54.7 ... 20.4 20.3 20.2 20.1
  * longitude       (x) float64 6kB 230.0 230.1 230.2 ... 299.7 299.8 299.9
    step            timedelta64[ns] 8B 00:00:00
    heightAboveSea  float64 8B 500.0
    valid_time      datetime64[ns] 8B 2025-09-06T15:00:39
Dimensions without coordinates: y, x
Data variables:
    unknown         (y, x) float32 980kB nan nan nan nan nan ... nan nan nan nan
Attributes:
    GRIB_edition:            2
    GRIB_centre:             161
    GRIB_centreDescription:  US NOAA Office of Oceanic and Atmospheric Research
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US NOAA Office of Oceanic and Atmospheric Research
    history:                 2026-01-08T11:32 GRIB to CDM+CF via cfgrib-0.9.1...

In [56]:
# Create z coordinate from 0 to 10000 in increments of 100
z_values = np.arange(0, 10001, 100)

# Expand the dataset to include z dimension (values broadcast uniformly across z)
ds['unknown_z'] = ds['unknown'].expand_dims(z=z_values)
for var in ["x", "y"]:
    ds = ds.isel({var: slice(None, None, 10)})
ds

<xarray.Dataset> Size: 1MB
Dimensions:         (y: 35, x: 70, z: 101)
Coordinates:
  * latitude        (y) float64 280B 54.99 54.0 53.0 52.0 ... 23.0 22.0 21.0
  * longitude       (x) float64 560B 230.0 231.0 232.0 ... 297.0 298.0 299.0
  * z               (z) int64 808B 0 100 200 300 400 ... 9700 9800 9900 10000
    step            timedelta64[ns] 8B 00:00:00
    heightAboveSea  float64 8B 500.0
    valid_time      datetime64[ns] 8B 2025-09-06T15:00:39
Dimensions without coordinates: y, x
Data variables:
    unknown         (y, x) float32 10kB nan nan nan -99.0 ... nan nan nan nan
    unknown_z       (z, y, x) float32 990kB nan nan nan -99.0 ... nan nan nan
Attributes:
    GRIB_edition:            2
    GRIB_centre:             161
    GRIB_centreDescription:  US NOAA Office of Oceanic and Atmospheric Research
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US NOAA Office of Oceanic and Atmospheric Research
    history:                 2026-01-08T11:32 GRIB to CDM+CF via cfgrib-0.9.1...

In [62]:
_ = sv.plot_gridded(ds=ds, slices={"unknown": {}})

/home/cmdavis4/projects/carlee/skyvista/skyvista/varspec.py:421: UserWarning: Points is not a float type. This can cause issues when transforming or applying filters. Casting to ``np.float32``. Disable this by passing ``force_float=False``.
  mesh = pv.StructuredGrid(coords["x"], coords["y"], coords["z"], **create_kwargs)


Widget(value='<iframe src="http://localhost:41769/index.html?ui=P_0x7f05ec4c60d0_9&reconnect=auto" class="pyvi…